# Three representative opinion states ($K=40$)

This notebook generates the three trajectory panels, the three terminal-distribution panels, and the combined six-panel figure used in the manuscript.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from monte_carlo import MonteCarloParameters, simulate_trajectory

figure_dir = repo_root / 'figures'
figure_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 0.9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


## 1. Parameters
Change the model parameters here. All three simulations use $K=40$.

In [ ]:
parameters = MonteCarloParameters(
    n=800,
    k=40,
    t_max=100,
    f_s=0.50,
    sigma_s=0.80,
    sigma_ns=0.10,
    gamma=4.0,
    d_c=0.50,
    x0_stubborn_positive=1.0,
    x0_stubborn_negative=-1.0,
    x0_nonstubborn_negative=-0.5,
    x0_nonstubborn_positive=0.5,
    seed=80,
    trajectory_seed=123,
    trajectories_per_bias_group=60,
)

cases = [
    {'roman': 'I',   'name': 'Bimodal polarized', 'alpha2': 0.03, 'color': '#6F4E9C'},
    {'roman': 'II',  'name': 'Biased unimodal',   'alpha2': 0.10, 'color': '#D9772B'},
    {'roman': 'III', 'name': 'Neutral unimodal',  'alpha2': 0.30, 'color': '#2B7BAA'},
]
parameters


## 2. Run the three Monte Carlo simulations

In [ ]:
results = []
for case in cases:
    result = simulate_trajectory(alpha2=case['alpha2'], parameters=parameters)
    results.append(result)
    print(
        f"{case['roman']}: {case['name']}, alpha2={case['alpha2']:.2f}, "
        f"B={result['B']:.3f}, S={result['S']:.3f}"
    )


## 3. Plotting functions

In [ ]:
negative_mean_color = '#2166AC'
positive_mean_color = '#B2182B'
total_mean_color = '#202020'

def style_axis(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(direction='out', labelsize=12.5, width=0.9, length=4)
    ax.xaxis.label.set_size(20)
    ax.yaxis.label.set_size(20)

def draw_trajectory(ax, result, case):
    ax.plot(result['times'], result['sampled_history'], color=case['color'], alpha=0.19, linewidth=0.44, rasterized=True)
    ax.plot(result['times'], result['negative_mean'], color=negative_mean_color, linewidth=2.05)
    ax.plot(result['times'], result['positive_mean'], color=positive_mean_color, linewidth=2.05)
    ax.plot(result['times'], result['total_mean'], color=total_mean_color, linewidth=2.2, linestyle=(0, (5, 2.5)))
    ax.set(xlim=(-3, 103), ylim=(-1.05, 1.05), xlabel=r'$t$', ylabel=r'$x_i(t)$')
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    style_axis(ax)

def draw_distribution(ax, result, case):
    x_grid = np.linspace(-1.0, 1.0, 1000)
    density = gaussian_kde(result['final_opinions'])(x_grid)
    ax.fill_between(x_grid, 0.0, density, color=case['color'], alpha=0.07, linewidth=0)
    ax.plot(x_grid, density, color=case['color'], linewidth=2.0)
    ax.set(xlim=(-1.08, 1.08), ylim=(0, None), xlabel=r'$x_i$', ylabel=r'$p(x_i)$')
    ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    style_axis(ax)

legend_handles = [
    Line2D([0], [0], color=negative_mean_color, lw=2.05, label=r'Negative-bias mean ($\eta_i<0$)'),
    Line2D([0], [0], color=positive_mean_color, lw=2.05, label=r'Positive-bias mean ($\eta_i>0$)'),
    Line2D([0], [0], color=total_mean_color, lw=2.2, linestyle=(0, (5, 2.5)), label='All non-stubborn mean'),
]


## 4. Generate the three trajectory panels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.5), dpi=160)
for ax, case, result in zip(axes, cases, results):
    draw_trajectory(ax, result, case)
    ax.set_title(f"{case['roman']}  {case['name']}\n" + rf"$\alpha_2={case['alpha2']:.2f}$", fontsize=13, fontweight='semibold')

fig.legend(handles=legend_handles, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.08), fontsize=15.5, handlelength=3.0, columnspacing=1.2)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(figure_dir / 'trajectory_states_K40_only.pdf', bbox_inches='tight')
fig.savefig(figure_dir / 'trajectory_states_K40_only.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Generate the three terminal-distribution panels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.3), dpi=160)
for ax, case, result in zip(axes, cases, results):
    draw_distribution(ax, result, case)
    ax.set_title(
        f"{case['roman']}  {case['name']}\n" + rf"$B={result['B']:.3f},\quad S={result['S']:.3f}$",
        fontsize=13,
        fontweight='semibold',
    )
fig.tight_layout()
fig.savefig(figure_dir / 'terminal_distributions_K40_only.pdf', bbox_inches='tight')
fig.savefig(figure_dir / 'terminal_distributions_K40_only.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Generate the combined six-panel manuscript figure
The $B$ and $S$ labels are positioned closer to the lower distribution panels.

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(12.8, 7.25), dpi=180,
    gridspec_kw={'left': 0.07, 'right': 0.985, 'bottom': 0.09, 'top': 0.775, 'wspace': 0.30, 'hspace': 0.60},
)

for column, (case, result) in enumerate(zip(cases, results)):
    draw_trajectory(axes[0, column], result, case)
    draw_distribution(axes[1, column], result, case)

    position = axes[0, column].get_position()
    x_center = 0.5 * (position.x0 + position.x1)
    fig.text(x_center, 0.975, case['roman'], ha='center', va='center', fontsize=16, fontweight='bold')
    fig.text(x_center, 0.940, case['name'], ha='center', va='center', fontsize=14, fontweight='semibold')
    fig.text(x_center, 0.912, rf"$\alpha_2={case['alpha2']:.2f}$", ha='center', va='center', fontsize=13)

    upper = axes[0, column].get_position()
    lower = axes[1, column].get_position()
    metric_y = 0.4 * upper.y0 + 0.6 * lower.y1
    fig.text(
        x_center, metric_y,
        rf"$B={result['B']:.3f},\quad S={result['S']:.3f}$",
        ha='center', va='center', fontsize=13,
    )

fig.legend(
    handles=legend_handles, loc='upper center', ncol=3, frameon=True,
    bbox_to_anchor=(0.5, 0.885), fontsize=15.5, handlelength=3.0, columnspacing=1.2,
)
fig.savefig(figure_dir / 'trajectory_states_K40_final.pdf', bbox_inches='tight')
fig.savefig(figure_dir / 'trajectory_states_K40_final.png', dpi=300, bbox_inches='tight')
plt.show()
